# habitable candidates of the crossmatched sample
- tabularray `longtblr` for `\input{}` inside the appendix landscape block
- exports to `report/figures/rocky_temperate_candidates_table.tex`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from report_plots import common, data, tables


In [ ]:
cme, input_table, matched = data.emc_crossmatch()
merger = data.build_param_filler()

e_cm_emc = data.enrich(matched, cme, merger)
e_emc = data.enrich(cme.catalog_table, cme, merger)
data.compute_is_rocky_temperate(e_cm_emc)
data.compute_is_rocky_temperate(e_emc)
data.rocky_temperate_summary(e_cm_emc=e_cm_emc, e_emc=e_emc)


In [ ]:
tex = tables.build_rocky_temperate_table_tex(
    e_cm_emc,
    common.FIGURES_DIR / "rocky_temperate_candidates_table.tex",
    caption=r"Rocky, temperate planet candidates in HPICxLIFEcat sample crossmatched with Exo-MerCat (see \cref{fig:rocky_temp_yield})",
    label="tab:rocky_temp_table",
)
print(tex)


## Export to CSV

In [ ]:
from datetime import date

# Keep the candidate definition identical between the enriched and published exports.
published = data.enrich(matched, cme, merger, disable_calculations=True)
candidate_mask = e_cm_emc["rocky_temp_status"] != ""
candidate_rows = [idx for idx, is_candidate in enumerate(candidate_mask) if is_candidate]

enriched_candidates = e_cm_emc[candidate_rows]
published_candidates = published[candidate_rows]

export_dir = common.ROOT / "report" / "data_exports"
export_dir.mkdir(parents=True, exist_ok=True)
export_date = date.today().isoformat()

enriched_candidates.write(
    export_dir / f"{export_date}-HPICxLIFEcat-habitable-enriched.csv",
    format="csv",
    overwrite=True,
)
published_candidates.write(
    export_dir / f"{export_date}-HPICxLIFEcat-habitable-published.csv",
    format="csv",
    overwrite=True,
)
e_cm_emc.write(
    export_dir / f"{export_date}-HPICxLIFEcat-enriched.csv",
    format="csv",
    overwrite=True,
)
published.write(
    export_dir / f"{export_date}-HPICxLIFEcat-published.csv",
    format="csv",
    overwrite=True,
)